# Hybrid Search Engine

We combine TF-IDF and Neural Embeddings to build a hybrid search system.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

c:\Users\Felhasználó\TextMiningAndNaturalLanguageProcessingHomeAssignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../Data/final_tweets.csv")

print(df.shape)
df.head()

(59184, 4)


,tweet_id,text,clean_text,company
0,277379,Another great set of @Delta flights thank to #...,another great set of flights thank to tmobilew...,delta
1,2488664,@AppleSupport why is my iPhone automatically g...,why is my iphone automatically going on mute,applesupport
2,2383194,@Uber_Support Since yesterday I haven't receiv...,since yesterday i havent received any update a...,uber_support
3,1610886,"@AmazonHelp There is no email from 2 days, jus...",there is no email from days just asked to wait...,amazonhelp
4,2763258,"@SouthwestAir thanks for responding, will call...",thanks for responding will call as soon as i g...,southwestair


In [4]:
search_df = df[["tweet_id", "clean_text", "company"]].copy()

search_df.head()

,tweet_id,clean_text,company
0,277379,another great set of flights thank to tmobilew...,delta
1,2488664,why is my iphone automatically going on mute,applesupport
2,2383194,since yesterday i havent received any update a...,uber_support
3,1610886,there is no email from days just asked to wait...,amazonhelp
4,2763258,thanks for responding will call as soon as i g...,southwestair


In [5]:
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(search_df["clean_text"])

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (59184, 10000)


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11876.65it/s]


Embedding model loaded


In [7]:
texts = search_df["clean_text"].tolist()

embeddings = model.encode(texts, show_progress_bar=True)

print("Embeddings shape:", embeddings.shape)


Batches: 100%|██████████| 1850/1850 [05:58<00:00,  5.17it/s]


Embeddings shape: (59184, 384)


In [8]:
def hybrid_search(query, top_k=5, alpha=0.5):
    # TF-IDF score
    query_tfidf = tfidf.transform([query])
    tfidf_scores = cosine_similarity(query_tfidf, tfidf_matrix).flatten()
    
    # Embedding score
    query_emb = model.encode([query])
    emb_scores = cosine_similarity(query_emb, embeddings).flatten()
    
    # Combine both scores
    hybrid_scores = alpha * tfidf_scores + (1 - alpha) * emb_scores
    
    # Top results
    top_indices = hybrid_scores.argsort()[::-1][:top_k]
    
    results = search_df.iloc[top_indices].copy()
    results["score"] = hybrid_scores[top_indices]
    
    return results

In [10]:
queries = [
    "flight delayed customer service",
    "refund for cancelled order",
    "app not working",
    "bad customer support",
    "lost baggage complaint"
]

for q in queries:
    print("=" * 80)
    print("Query:", q)
    display(hybrid_search(q, top_k=5, alpha=0.5))

Query: flight delayed customer service


,tweet_id,clean_text,company,score
2472,1799160,flight was already delayed on top of it poor c...,americanair,0.834247
30389,565503,why is flight delayed,americanair,0.715450
51365,1370386,when your flight is delayed,southwestair,0.711692
36349,2098673,my flight has been delayed hours and no custom...,americanair,0.687231
46536,2258648,been there done that with customer service thr...,americanair,0.657047


Query: refund for cancelled order


,tweet_id,clean_text,company,score
44897,1736159,order cancelled,amazonhelp,0.792641
29980,242928,cancelled my order,amazonhelp,0.788732
4601,2192481,you cancelled the order not me,amazonhelp,0.728759
22077,2283942,i cancelled an order and it successfully cance...,amazonhelp,0.679037
25933,581913,ur representative misguided me against the amo...,amazonhelp,0.643600


Query: app not working


,tweet_id,clean_text,company,score
53628,2884478,yet again app not working,uber_support,0.825137
49472,2216875,not working,amazonhelp,0.712620
5001,1858429,not working,applesupport,0.712620
2469,2843793,i uninstalled the app and now its working,amazonhelp,0.690184
8743,1332295,my app store is not working ios,applesupport,0.680252


Query: bad customer support


,tweet_id,clean_text,company,score
1329,1098833,worst customer support ever,delta,0.722607
287,382575,customer for years worst customer support ever,amazonhelp,0.695835
19359,2552007,just lost due to bad customer support represen...,amazonhelp,0.672575
42959,1004647,i never have a bad day when calling your custo...,amazonhelp,0.558891
11440,1149433,done can i get a call from customer support an...,americanair,0.530663


Query: lost baggage complaint


,tweet_id,clean_text,company,score
57075,1906961,could you advise me on my lost baggage claim,delta,0.637330
2331,1111612,why cant you locate lost baggage are you kiddi...,southwestair,0.587181
38200,2799767,next issue will be my baggage find it please n...,americanair,0.529451
15873,2104057,been given a baggage information form re delay...,delta,0.529324
50273,630078,the lost baggage process and the people who ma...,delta,0.522980


The hybrid search results show that combining TF-IDF and neural embedding similarity gives relevant results across different types of customer complaints. For keyword-specific queries such as **“refund for cancelled order”** and **“app not working”**, the retrieved tweets closely match the exact query terms. For broader complaint-based queries such as **“bad customer support”** and **“lost baggage complaint”**, the method also finds semantically related tweets even when the wording is not exactly the same.